# 전처리 효과 빠른 확인 — ResNet18

**전처리 파이프라인**
1. 2단계 선별(현환님 전처리) usable → 선별기 자체의 베드 화소 지도로 베드를 회색 처리
2. 2단계 선별 보류 → 형상 v2가 경계를 찾은 사진만 형상 v2 마스크로 회색 처리해 재활용, 나머지는 보류

| 실험 | 사진 | 장수 (정상 / 언더필) |
|---|---|---|
| **base** 선별만 | usable, 원본 크롭 | 2652 (1351 / 1301) |
| **pipe** 전처리 파이프라인 | usable(마스킹) + 재활용 743장 | 3395 (1730 / 1665) |

- 빠른 확인용이라 ResNet18, 시드 0 하나만 돌린다. 나머지(고정 크롭, 512, finetune 20 epoch, 시간 블록 5-fold, 30프레임 buffer 분할표)는 지난 실험과 같다.
- 비교는 세 가지 사진 집합으로 본다: 각자 사진 / 공통 사진(base의 사진) / 재활용 사진만.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `three_conditions_bundle_20261005.zip`: **새로 올릴 것** (학습 스크립트 `--mask-csv` 추가본, 분할표, 사진 목록, 마스크 PNG)

노트북은 **파일 > 노트북 업로드**로 연다. 4번 셀을 누르고 **런타임 > 이후 셀 실행**을 하면 학습 → 결과 → CSV 묶기 → 런타임 반납까지 한 번에 한다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/preproc_quick_resnet18_20261005"
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "three_conditions_bundle_20261005.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
`--mask-csv`가 보이는 줄, 목록 줄 수(제목 1줄 포함) `keep_cond1.csv` 2653, `keep_cond3.csv` 3396, 마스크 PNG 3395장, 원본 4000장이 나와야 한다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/three_conditions_bundle_20261005.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/three_conditions_bundle_20261005.zip -d /content/code
!grep -m1 -o -- "--mask-csv" /content/code/train_pilot_classifier.py
!wc -l < /content/code/keep_cond1.csv; wc -l < /content/code/keep_cond3.csv
!find /content/code/masks -name "*.png" | wc -l
!find /content/orig -name "*.jpg" | wc -l

## 4. 본 실행 (base → pipe)
- 시작 몇 분 뒤 나와야 하는 줄
  - base: `keep-list keep_cond1.csv: held 1348 images`
  - pipe: `keep-list keep_cond3.csv: held 605 images`, `mask-csv masks_cond3.csv: 3395 of 3395 images masked`
- 학습 로그는 화면 대신 Drive의 `results/preproc_quick_resnet18_20261005/<실험>_seed0.log`에 저장한다. fold가 끝날 때마다 한 줄씩만 화면에 나온다.
- **끊겼을 때:** 이 셀을 다시 실행하면 끝난 실험은 건너뛰고, 하던 실험은 처음부터 다시 한다.
- **시간이 모자라면:** 셀을 멈추고(■) 5번부터 실행한다. 끝난 실험·fold까지 보여 준다.

In [ ]:
import shutil
C = "/content/code"
BASE = (f"--folds-csv {C}/folds.csv --pilot-root /content/orig --input-column source "
        "--fixed-crop 0 250 1000 1250 --img-size 512 --modes finetune --fast --arch resnet18 --folds 4 0 3 1 2")
EXPS = {"base_selector": f"--keep-list {C}/keep_cond1.csv",
        "pipe_mask_recycle": f"--keep-list {C}/keep_cond3.csv --mask-csv {C}/masks_cond3.csv"}
SEEDS = [0]

def run(exp, seed):
    out = f"{RESULTS}/{exp}_seed{seed}"
    if os.path.exists(f"{out}/summary.csv"):
        print("이미 끝남:", out)
        return
    shutil.rmtree(out, ignore_errors=True)   # 반만 끝난 폴더 정리
    log = f"{RESULTS}/{exp}_seed{seed}.log"
    print(f"\n===== {exp} seed {seed}  (log: {log})")
    !PYTHONUNBUFFERED=1 python {C}/train_pilot_classifier.py {BASE} {EXPS[exp]} --seeds {seed} --output "{out}" > "{log}" 2>&1; grep -E "keep-list|mask-csv|fold|Error|error" "{log}" | grep -v "^ " | tail -n 20

for SEED in SEEDS:
    for EXP in EXPS:
        run(EXP, SEED)

## 5. 결과
- **own:** 실험마다 자기 사진 전체 기준. `n`이 판정한 장수다.
- **common(c1):** base의 사진만으로 비교한다. 마스킹·재활용이 원래 사진 성능을 바꾸는지 본다.
- **recycled:** pipe가 새로 판정하게 된 사진의 정확도다. base는 이 사진들을 보류한다.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd, json, glob
keep1 = set(pd.read_csv(f"{C}/keep_cond1.csv", encoding="utf-8-sig").source)
rows, folds = [], []
for exp in EXPS:
    for seed in SEEDS:
        out = f"{RESULTS}/{exp}_seed{seed}"
        for p in sorted(glob.glob(f"{out}/runs/finetune/seed{seed}/fold*/metrics.json")):
            m = json.load(open(p))
            folds.append({"exp": exp, "seed": seed, "fold": m["fold"], "acc": m["val_accuracy"]})
        if not os.path.exists(f"{out}/oof_predictions.csv"):
            continue
        d = pd.read_csv(f"{out}/oof_predictions.csv", encoding="utf-8-sig")
        v = pd.read_csv(f"{out}/vote_metrics.csv", encoding="utf-8-sig").iloc[0]
        for part, sub in (("own", d), ("common(c1)", d[d.source.isin(keep1)]), ("recycled", d[~d.source.isin(keep1)])):
            if len(sub) == 0:
                continue
            fp = int(((sub.label_id == 0) & (sub.pred == 1)).sum()); fn = int(((sub.label_id == 1) & (sub.pred == 0)).sum())
            rows.append({"exp": exp, "seed": seed, "set": part, "n": len(sub), "n_normal": int((sub.label_id == 0).sum()),
                         "n_underfill": int((sub.label_id == 1).sum()), "acc": 1 - (fp + fn) / len(sub), "FP": fp, "FN": fn,
                         "vote_acc": v.vote_accuracy if part == "own" else None})
if folds:
    print("[fold별 정확도]\n" + pd.DataFrame(folds).pivot_table(index=["exp", "seed"], columns="fold", values="acc").round(3).to_string())
if rows:
    print("\n[조건 x 사진 집합]\n" + pd.DataFrame(rows).round(4).to_string(index=False))

## 6. 결과 CSV 묶기
가중치를 뺀 CSV·로그만 Drive `results/preproc_quick_resnet18_20261005_csv.zip`으로 묶는다. 이 zip을 Claude에게 주면 연속 보류 시간 같은 공정 단위 지표까지 계산해 준다.

In [ ]:
import zipfile
ZIP = f"{DRIVE_DIR}/results/preproc_quick_resnet18_20261005_csv.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt", ".log")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 7. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()